# 🛠️ Function Calling Fine-tuning Guide
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/togethercomputer/together-cookbook/blob/main/Finetuning/Function_Calling_Finetuning.ipynb)

Function calling fine-tuning teaches models to reliably invoke tools and structured functions in response to user queries. This is essential for building agents that can interact with APIs, query databases, and perform actions in the real world.

This notebook provides a step-by-step guide to fine-tuning a model for function calling using the Together AI platform. We'll use a subset of the [glaive-function-calling-v2](https://huggingface.co/datasets/togethercomputer/glaive-function-calling-v2-formatted) dataset to train a model that can accurately select and call the right tools with correct arguments.

We will cover:

1. **Dataset Exploration:** Loading and understanding the function calling dataset.
2. **Data Transformation:** Converting the dataset to Together AI's function calling fine-tuning format.
3. **Data Upload:** Validating and uploading the prepared dataset to Together AI.
4. **Fine-tuning Job Launch:** Configuring and starting a LoRA fine-tuning job.
5. **Job Monitoring:** Checking the status and progress of your fine-tuning job.
6. **Inference:** Comparing the base model vs. fine-tuned model on function calling tasks.

By following this guide, you'll learn how to create models that reliably call the right tools with correct arguments using Together AI.

## Setup and Installation
---
First, install the necessary Python libraries:
- `together`: The official Together AI Python client for interacting with the API.
- `datasets`: A library from Hugging Face for easily downloading and manipulating datasets.
- `tqdm`: For progress bars.

In [1]:
!uv pip install -qU together datasets tqdm

In [2]:
import os
import json
import uuid
from together import Together
from datasets import load_dataset
from tqdm.auto import tqdm

# Setup Together AI client
TOGETHER_API_KEY = os.getenv("TOGETHER_API_KEY")
WANDB_API_KEY = os.getenv("WANDB_API_KEY")  # Optional: for logging to WandB

client = Together(api_key=TOGETHER_API_KEY)

## 1. Dataset Exploration
---
We'll use the [glaive-function-calling-v2-formatted](https://huggingface.co/datasets/togethercomputer/glaive-function-calling-v2-formatted) dataset, a curated collection of function calling conversations. The dataset contains:
- **112,000 training examples** of conversations that include function calling
- **1,000 test examples**

Each example includes a conversation with a system prompt, user query, and assistant responses that invoke functions, along with the available tool definitions.

In [3]:
# Load the dataset
dataset = load_dataset("togethercomputer/glaive-function-calling-v2-formatted")
print(dataset)

README.md:   0%|          | 0.00/662 [00:00<?, ?B/s]

data/train-00000-of-00002.parquet:   0%|          | 0.00/99.6M [00:00<?, ?B/s]

data/train-00001-of-00002.parquet:   0%|          | 0.00/99.3M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.66M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/111944 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'messages', 'tools'],
        num_rows: 111944
    })
    test: Dataset({
        features: ['text', 'messages', 'tools'],
        num_rows: 1000
    })
})


In [10]:
# Examine a sample from the training set
sample = dataset["train"][1]
print("Keys in each sample:", list(sample.keys()))

# The dataset stores messages/tools as JSON strings — parse them
messages = json.loads(sample["messages"]) if isinstance(sample["messages"], str) else sample["messages"]
tools = json.loads(sample["tools"]) if isinstance(sample["tools"], str) else sample["tools"]

print(f"\nNumber of messages: {len(messages)}")
print(f"Number of tools: {len(tools)}")

Keys in each sample: ['text', 'messages', 'tools']

Number of messages: 5
Number of tools: 1


In [11]:
# Look at the messages in this sample
messages = json.loads(sample["messages"])

for msg in messages:
    role = msg["role"]
    content = msg["content"]
    # Truncate long system prompts for display
    if role == "system" and len(content) > 200:
        content = content[:200] + "..."
    print(f"[{role}]: {content}\n")

[system]: You are a helpful assistant with access to the following functions. Use them if required -
[
  {
    "type": "function",
    "function": {
      "name": "send_email",
      "description": "Send an ema...

[user]: I need to send an email to my boss. Can you help me with that?

[assistant]: Of course, I can help you with that. Could you please provide me with the recipient's email address, the subject of the email, and the message you want to send?

[user]: Sure, the recipient's email is boss@company.com. The subject is "Project Update" and the message is "Dear Boss, I have completed the project as per the given deadline. I have attached the final report for your review. Regards, [User's Name]".

[assistant]: [
  {
    "name": "send_email",
    "arguments": {
      "recipient": "boss@company.com",
      "subject": "Project Update",
      "message": "Dear Boss, I have completed the project as per the given deadline. I have attached the final report for your review. Regards, [Use

In [12]:
# Look at the tool definitions
tools = json.loads(sample["tools"]) if isinstance(sample["tools"], str) else sample["tools"]
print("Tool definitions:")
print(json.dumps(tools, indent=2))

Tool definitions:
[
  {
    "type": "function",
    "function": {
      "name": "send_email",
      "description": "Send an email to a recipient",
      "parameters": {
        "type": "object",
        "properties": {
          "recipient": {
            "type": "string",
            "description": "The email address of the recipient"
          },
          "subject": {
            "type": "string",
            "description": "The subject of the email"
          },
          "message": {
            "type": "string",
            "description": "The body of the email message"
          }
        },
        "required": [
          "recipient",
          "subject",
          "message"
        ]
      }
    }
  }
]


Notice the current format of this dataset:
- The **system message** contains the tool definitions embedded in the text content
- The **assistant messages** contain function calls as JSON strings in the `content` field
- The **tools** are stored as a separate column with proper OpenAI-style schemas

For Together AI's function calling fine-tuning format, we need to restructure this so that:
- Assistant messages use the `tool_calls` field (not `content`) for function invocations
- Tool results use the `tool` role
- The `tools` field is at the top level of each example alongside `messages`

## 2. Data Transformation to Function Calling Format
---
Together AI's function calling fine-tuning requires a specific format where tool invocations use the `tool_calls` field in assistant messages.

### Required Format
```json
{
  "messages": [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What's the weather in SF?"},
    {"role": "assistant", "tool_calls": [
      {
        "id": "call_abc123",
        "type": "function",
        "function": {
          "name": "getCurrentWeather",
          "arguments": "{\"location\": \"San Francisco, CA\"}"
        }
      }
    ]},
    {"role": "tool", "content": "{\"temperature\": \"65\", \"unit\": \"fahrenheit\"}"}
  ],
  "tools": [
    {
      "type": "function",
      "function": {
        "name": "getCurrentWeather",
        "description": "Get the current weather in a given location",
        "parameters": {
          "type": "object",
          "properties": {
            "location": {"type": "string", "description": "The city and state"}
          },
          "required": ["location"]
        }
      }
    }
  ]
}
```

### Key Requirements:
- Assistant messages with function calls use `tool_calls` (not `content`)
- Each tool call needs an `id`, `type: "function"`, and `function` with `name` and `arguments` (as a JSON string)
- Tool responses use the `tool` role with the result in `content`

We'll also **subsample 200 training and 50 validation examples** since this is a demo. For production use cases, you'd use significantly more data.

In [13]:
def parse_tool_calls(content: str) -> list[dict] | None:
    """
    Parse assistant content that contains function calls as JSON.
    Returns a list of tool_calls in the OpenAI format, or None if content
    is not a function call.
    """
    content = content.strip()
    try:
        parsed = json.loads(content)
    except (json.JSONDecodeError, TypeError):
        return None

    # The dataset stores function calls as a JSON array of {name, arguments}
    if isinstance(parsed, list):
        calls = parsed
    elif isinstance(parsed, dict) and "name" in parsed:
        calls = [parsed]
    else:
        return None

    tool_calls = []
    for call in calls:
        if "name" not in call:
            continue
        arguments = call.get("arguments", {})
        tool_calls.append({
            "id": f"call_{uuid.uuid4().hex[:8]}",
            "type": "function",
            "function": {
                "name": call["name"],
                "arguments": json.dumps(arguments) if isinstance(arguments, dict) else str(arguments),
            },
        })

    return tool_calls if tool_calls else None

In [15]:
SYSTEM_PROMPT = "You are a helpful assistant with access to tools. Use them when appropriate to answer user questions."


def convert_to_fc_format(example: dict) -> dict | None:
    """
    Convert a dataset example to Together AI's function calling fine-tuning format.

    Transforms assistant messages containing JSON function calls into proper
    tool_calls format, and replaces the embedded system prompt with a clean one.

    Args:
        example: A single example from the dataset with 'messages' and 'tools' fields.

    Returns:
        Dictionary in function calling fine-tuning format, or None if conversion fails.
    """
    tools = example.get("tools", [])
    if isinstance(tools, str):
        tools = json.loads(tools)
    if not tools:
        return None

    raw_messages = example.get("messages", [])
    if isinstance(raw_messages, str):
        raw_messages = json.loads(raw_messages)

    new_messages = []
    skip_next = False

    for i, msg in enumerate(raw_messages):
        if skip_next:
            skip_next = False
            continue

        role = msg["role"]
        content = msg["content"]

        if role == "system":
            # Replace the embedded system prompt with a clean one
            new_messages.append({"role": "system", "content": SYSTEM_PROMPT})

        elif role == "assistant":
            tool_calls = parse_tool_calls(content)
            if tool_calls:
                # This is a function call — use tool_calls field instead of content
                new_messages.append({"role": "assistant", "tool_calls": tool_calls})

                # Check if the next message is a follow-up from the user containing
                # a function response (the dataset sometimes embeds tool results
                # in the next user or assistant message)
                if i + 1 < len(raw_messages):
                    next_msg = raw_messages[i + 1]
                    # If the next message looks like a function response, add it as tool role
                    if next_msg["role"] == "user" and next_msg["content"].strip().startswith("{"):
                        try:
                            json.loads(next_msg["content"])
                            new_messages.append({
                                "role": "tool",
                                "content": next_msg["content"],
                            })
                            skip_next = True
                        except json.JSONDecodeError:
                            pass
            else:
                # Regular assistant text response
                new_messages.append({"role": "assistant", "content": content})

        elif role == "user":
            new_messages.append({"role": "user", "content": content})

    # Ensure the conversation has at least a user and assistant message
    roles = [m["role"] for m in new_messages]
    if "user" not in roles or "assistant" not in roles:
        return None

    # Ensure at least one assistant message has tool_calls (this is function calling data)
    has_tool_call = any("tool_calls" in m for m in new_messages if m.get("role") == "assistant")
    if not has_tool_call:
        return None

    return {"messages": new_messages, "tools": tools}

In [16]:
# Test the conversion on our sample
converted = convert_to_fc_format(sample)

print("Converted format:")
print(json.dumps(converted, indent=2))

Converted format:
{
  "messages": [
    {
      "role": "system",
      "content": "You are a helpful assistant with access to tools. Use them when appropriate to answer user questions."
    },
    {
      "role": "user",
      "content": "I need to send an email to my boss. Can you help me with that?"
    },
    {
      "role": "assistant",
      "content": "Of course, I can help you with that. Could you please provide me with the recipient's email address, the subject of the email, and the message you want to send?"
    },
    {
      "role": "user",
      "content": "Sure, the recipient's email is boss@company.com. The subject is \"Project Update\" and the message is \"Dear Boss, I have completed the project as per the given deadline. I have attached the final report for your review. Regards, [User's Name]\"."
    },
    {
      "role": "assistant",
      "tool_calls": [
        {
          "id": "call_09c5c9e4",
          "type": "function",
          "function": {
            "nam

In [17]:
# Convert and subsample the dataset
# We use a small subset for this demo — scale up for production use cases
TRAIN_SIZE = 200
VAL_SIZE = 50

print("Converting training examples...")
train_data = []
for example in tqdm(dataset["train"]):
    converted = convert_to_fc_format(example)
    if converted:
        train_data.append(converted)
    if len(train_data) >= TRAIN_SIZE:
        break

print(f"Converted {len(train_data)} training examples")

print("\nConverting validation examples...")
val_data = []
# Use the test split for validation
for example in tqdm(dataset["test"]):
    converted = convert_to_fc_format(example)
    if converted:
        val_data.append(converted)
    if len(val_data) >= VAL_SIZE:
        break

print(f"Converted {len(val_data)} validation examples")

Converting training examples...


  0%|          | 0/111944 [00:00<?, ?it/s]

Converted 200 training examples

Converting validation examples...


  0%|          | 0/1000 [00:00<?, ?it/s]

Converted 50 validation examples


In [18]:
# Save to JSONL files
def save_jsonl(data: list, filepath: str):
    """Save a list of dictionaries to a JSONL file."""
    with open(filepath, "w") as f:
        for item in data:
            f.write(json.dumps(item) + "\n")
    print(f"Saved {len(data)} examples to {filepath}")

save_jsonl(train_data, "fc_train.jsonl")
save_jsonl(val_data, "fc_val.jsonl")

Saved 200 examples to fc_train.jsonl
Saved 50 examples to fc_val.jsonl


## 3. Upload Data to Together AI
---
Now we'll validate and upload our prepared datasets to Together AI. The `check=True` parameter validates the file format before uploading.

In [20]:
# Upload training file
print("Uploading training file...")
train_file = client.files.upload("fc_train.jsonl", check=True)
print(f"Training file ID: {train_file.id}")

Uploading training file...


Validating file: 200 lines [00:00, 69713.35 lines/s]
Uploading file fc_train.jsonl: 100%|██████████| 248k/248k [00:00<00:00, 814kB/s]


Training file ID: file-f253dccc-9d68-4d01-a9e3-efb02de74206


In [21]:
# Upload validation file
print("Uploading validation file...")
val_file = client.files.upload("fc_val.jsonl", check=True)
print(f"Validation file ID: {val_file.id}")

Uploading validation file...


Validating file: 50 lines [00:00, 63859.68 lines/s]
Uploading file fc_val.jsonl: 100%|██████████| 58.6k/58.6k [00:00<00:00, 281kB/s]


Validation file ID: file-655272d8-0ee4-4ece-9a3f-25a4776baf41


## 4. Launch Fine-tuning Job
---
Now we'll launch a LoRA fine-tuning job for function calling.

### Key Parameters

| Parameter | Description | Our Value |
|-----------|-------------|-----------|
| `model` | Base model to fine-tune | `Qwen/Qwen3-8B` |
| `lora` | Use LoRA (recommended) | `True` |
| `n_epochs` | Training epochs | `3` |
| `learning_rate` | Weight update rate | `1e-5` |
| `n_checkpoints` | Checkpoints to save | `1` |
| `suffix` | Custom model name suffix | `fc-demo` |

> **Note:** These hyperparameters are tuned for this small demo dataset. For production, you'd use more data and may want to adjust epochs and learning rate accordingly.

🔗 See the [Function Calling Fine-tuning documentation](https://docs.together.ai/docs/function-calling-fine-tuning) for the full list of supported models and parameters.

In [22]:
# Launch the LoRA fine-tuning job
ft_response = client.fine_tuning.create(
    training_file=train_file.id,
    validation_file=val_file.id,
    model="Qwen/Qwen3-8B",
    lora=True,
    n_epochs=3,
    learning_rate=1e-5,
    n_checkpoints=1,
    suffix="fc-demo",
    wandb_api_key=WANDB_API_KEY,
)

print(f"Fine-tuning job created!")
print(f"Job ID: {ft_response.id}")

message="train_on_inputs is not set for SFT training, it will be set to 'auto'"
message="train_on_inputs is not set for SFT training, it will be set to 'auto'"


Fine-tuning job created!
Job ID: ft-05f3fbe3-bd07


## 5. Monitor Fine-tuning Job
---
You can monitor your fine-tuning job's progress using the Together AI API or the [dashboard](https://api.together.ai/jobs).

Your job will progress through several states: Pending → Queued → Running → Uploading → Completed.

Available methods:
- `client.fine_tuning.retrieve(id)` — Get job status
- `client.fine_tuning.list_events(id=job_id)` — Get job logs
- `client.fine_tuning.cancel(id=job_id)` — Cancel a job
- `client.fine_tuning.list()` — List all jobs

In [23]:
# Check the status of the fine-tuning job
status = client.fine_tuning.retrieve(ft_response.id)
print(f"Job Status: {status.status}")

Job Status: pending


In [24]:
# View training logs/events
events = client.fine_tuning.list_events(id=ft_response.id)
for event in events.data:
    print(event.message)

Fine-tuning request created


In [26]:
# Wait for job completion (optional — you can also check the dashboard)
import time

while True:
    status = client.fine_tuning.retrieve(ft_response.id)
    print(f"Status: {status.status}")

    if status.status in ["completed", "failed", "cancelled"]:
        break

    time.sleep(60)  # Check every minute

print(f"\nFinal status: {status.status}")

Status: completed

Final status: completed


🔗 You can also monitor your job on the [Together AI Fine-tuning Dashboard](https://api.together.ai/jobs) and view WandB logs if you provided an API key.

## 6. Inference — Base Model vs. Fine-tuned Model
---
Once training is complete, let's compare the base model against our fine-tuned model on function calling tasks. We'll define a set of test tools and queries to see how each model performs.

In [31]:
# Get the fine-tuned model name
finetuned_model = status.model_output_name
print(f"Fine-tuned model: {finetuned_model}")

Fine-tuned model: zainhas/Qwen3-8B-fc-demo-792ac632


In [32]:
# Define test tools and queries
test_tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a location",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "The city and state, e.g. San Francisco, CA",
                    },
                    "unit": {
                        "type": "string",
                        "enum": ["celsius", "fahrenheit"],
                        "description": "Temperature unit",
                    },
                },
                "required": ["location"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "search_restaurants",
            "description": "Search for restaurants in a given area",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "The area to search in",
                    },
                    "cuisine": {
                        "type": "string",
                        "description": "Type of cuisine to filter by",
                    },
                    "price_range": {
                        "type": "string",
                        "enum": ["$", "$$", "$$$", "$$$$"],
                        "description": "Price range filter",
                    },
                },
                "required": ["location"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "calculate_tip",
            "description": "Calculate the tip amount for a bill",
            "parameters": {
                "type": "object",
                "properties": {
                    "bill_amount": {
                        "type": "number",
                        "description": "The total bill amount",
                    },
                    "tip_percentage": {
                        "type": "number",
                        "description": "The percentage of tip to give",
                    },
                },
                "required": ["bill_amount", "tip_percentage"],
            },
        },
    },
]

test_queries = [
    "What's the weather like in Tokyo right now?",
    "Find me some Italian restaurants in downtown Chicago",
    "I have a bill of $85.50 and want to leave a 20% tip. How much should I tip?",
]

In [33]:
def test_function_calling(model_name: str, query: str, tools: list) -> dict:
    """
    Test a model's function calling capability.

    Args:
        model_name: Together AI model identifier.
        query: User query to send.
        tools: List of tool definitions.

    Returns:
        Dictionary with the model's response details.
    """
    response = client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": query},
        ],
        tools=tools,
        max_tokens=512,
    )

    choice = response.choices[0]
    result = {"content": choice.message.content}

    if choice.message.tool_calls:
        result["tool_calls"] = [
            {
                "function": tc.function.name,
                "arguments": tc.function.arguments,
            }
            for tc in choice.message.tool_calls
        ]

    return result

Below we compare the fine-tuned model against the base model on our test queries. We expect the fine-tuned model to more reliably select the correct tool and provide well-formed arguments.

In [ ]:
# Compare base model vs fine-tuned model
models_to_test = {
    "Base model (Qwen3-8B)": "Qwen/Qwen3-8B",
    "Fine-tuned model": finetuned_model,
}

for query in test_queries:
    print(f"Query: {query}")
    print("=" * 60)

    for label, model_id in models_to_test.items():
        print(f"\n{label}:")
        print("-" * 40)
        result = test_function_calling(model_id, query, test_tools)

        if "tool_calls" in result:
            for tc in result["tool_calls"]:
                print(f"  Tool: {tc['function']}")
                print(f"  Args: {tc['arguments']}")
        else:
            content = result["content"] or "(no response)"
            print(f"  Response: {content[:200]}")

    print("\n")

## Summary
---
In this notebook, we demonstrated how to:

1. **Load and explore** a function calling dataset from Hugging Face
2. **Transform data** into Together AI's function calling format with `tool_calls`, `tool` roles, and top-level `tools`
3. **Upload and validate** the dataset using the Together AI Python client
4. **Launch a LoRA fine-tuning job** on Qwen3-8B for function calling
5. **Monitor training progress** via the API and dashboard
6. **Compare inference** between the base model and fine-tuned model

### Key Takeaways
- Function calling fine-tuning requires assistant messages to use the `tool_calls` field with `id`, `type`, and `function` (name + arguments as JSON string)
- LoRA fine-tuning is recommended for function calling — it's faster, cheaper, and enables serverless inference
- Even a small subset of training data (200 examples) can improve a model's tool selection and argument formatting
- The `tools` field at the top level of each example defines the available functions for that conversation

### Next Steps
- Scale up to more training data for improved reliability
- Try DPO preference tuning to teach the model when *not* to call tools
- Experiment with different base models (Qwen3-8B, Qwen2.5-72B-Instruct, etc.)
- Deploy your fine-tuned model on a [dedicated endpoint](https://docs.together.ai/docs/dedicated-endpoints) for production use

🔗 For more details, see the [Function Calling Fine-tuning documentation](https://docs.together.ai/docs/function-calling-fine-tuning).